# XSD 与 SOXX 历史源覆盖预检

仅收集两只 ETF 的完整原始响应、公司行动和 NAS 备份。另一个冻结阶段才能定义可交易特征。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,math
from pathlib import Path
from datetime import datetime,timezone,date
from zoneinfo import ZoneInfo
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_xsd_semiconductor_breadth_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
raw=root/'data/snapshots/stage1_xsd_semiconductor_breadth_source_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 symbol=spec['symbol'];body=raw/(symbol+'.json');headers=raw/(symbol+'.headers');started=datetime.now(timezone.utc).isoformat();r=plan['request']
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(r['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'symbol':symbol,'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'symbol':symbol,'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>r['max_response_bytes_each']:failure='transport_or_size_failure';break
 try:
  payload=json.loads(body.read_text());chart=payload['chart'];assert chart['error'] is None;results=chart['result'];assert len(results)==1;result=results[0];meta=result['meta'];assert meta['symbol']==symbol
  times=result['timestamp'];ind=result['indicators'];close=ind['quote'][0]['close'];assert len(times)==len(close)>3000
  dates=[datetime.fromtimestamp(t,timezone.utc).astimezone(ZoneInfo('America/New_York')).date() for t in times];assert len(dates)==len(set(dates))
  valid=[(d,v) for d,v in zip(dates,close) if v is not None and math.isfinite(float(v)) and float(v)>0]
  assert valid and min(d for d,v in valid)<date(2011,1,3) and max(d for d,v in valid)>date(2026,10,7)
  adj=ind.get('adjclose');adj_len=len(adj[0]['adjclose']) if adj else 0
  check={'symbol':symbol,'status':'identity_and_full_span_passed','raw_sha256':receipt['raw_sha256'],'raw_bytes':receipt['raw_bytes'],'exchange_timezone':meta.get('exchangeTimezoneName'),'timestamp_rows':len(times),'positive_close_rows':len(valid),'first_positive_date':str(min(d for d,v in valid)),'last_positive_date':str(max(d for d,v in valid)),'adjusted_close_rows':adj_len,'event_categories':{k:len(v) for k,v in (result.get('events') or {}).items()},'duplicate_dates':0}
  checks.append(check)
 except Exception as exc:
  failure='source_identity_or_coverage_failure';checks.append({'symbol':symbol,'status':failure,'error':type(exc).__name__+':'+str(exc)[:150],'raw_sha256':receipt['raw_sha256']});break
 if i+1<len(plan['exact_requests']):time.sleep(r['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='xsd-soxx-source-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='hyg-lqd-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'XSD_SOXX_current_chart_raw_full_span_source_received_PIT_pending' if failure is None and len(checks)==2 else 'bounded_XSD_SOXX_source_probe_partial_or_failed','failure':failure,'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_corporate_action_vintage_qualified':False}
(card/'source_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'checks':checks,'NAS_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'XSD_SOXX_current_chart_raw_full_span_source_received_PIT_pending', 'failure': None, 'GETs': 2, 'checks': [{'symbol': 'XSD', 'status': 'identity_and_full_span_passed', 'raw_sha256': 'f6df2827bd64fb46a198863edcbb134f901fc3e595e4a8d2b8618e5c5fd11f5c', 'raw_bytes': 453863, 'exchange_timezone': 'America/New_York', 'timestamp_rows': 4218, 'positive_close_rows': 4218, 'first_positive_date': '2010-01-04', 'last_positive_date': '2026-10-09', 'adjusted_close_rows': 4218, 'event_categories': {'dividends': 67, 'splits': 1}, 'duplicate_dates': 0}, {'symbol': 'SOXX', 'status': 'identity_and_full_span_passed', 'raw_sha256': '3439ce09ac74780dd6713f278176b92440d0953b8d126c8de3e3c546e9a64b29', 'raw_bytes': 467650, 'exchange_timezone': 'America/New_York', 'timestamp_rows': 4218, 'positive_close_rows': 4218, 'first_positive_date': '2010-01-04', 'last_positive_date': '2026-10-09', 'adjusted_close_rows': 4218, 'event_categories': {'dividends': 65, 'splits': 1}, 'duplicate_dates': 0}], 'NAS_sha25

# XSD 与 SOXX 半导体领导结构源预检

State Street 的 XSD 跟踪改良等权半导体指数，2006 年成立；指数提供与 SOXX 不同的行业内部权重视角。本卡只验证历史源覆盖，不定义或筛选特征。

Yahoo 固定历史请求各返回 4,218 个有效日收盘，2010-01-04 至 2026-10-09，日期无重复；XSD 有 67 次分红和 1 次拆分，SOXX 有 65 次分红和 1 次拆分。两份原始响应、头文件与接收时钟已归档至 NAS，并完成远端 SHA-256 与逐文件回读。NAS 归档 SHA-256：`fbcf034d8a1b0b17a32ec2f6ac290e4c0a2605a4efc83371bb0941e308a2c52c`。

下一阶段先核对拆分与分红调整是否在当时可知、映射到冻结 Stage1 的 Qlib 信号日，并与原 39 项中的行业/交叉宽度特征做训练期冗余检查。只有不同且来源时间合格，才冻结单坐标 H5 信息检验；通过后才单独冻结 0bp 原生账户。

本卡 0 拟合、0 账户，没有 SOXX 超额结论。历史调整收盘及公司行动是今日回溯快照，首次公开和修订时钟待核实。

来源：[State Street XSD](https://www.ssga.com/us/en/individual/etfs/state-street-spdr-sp-semiconductor-etf-xsd)，[S&P 半导体精选行业指数](https://www.spglobal.com/spdji/en/indices/equity/sp-semiconductors-select-industry-index/)。